# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook hoàn chỉnh để thực thi toàn bộ quy trình thí nghiệm theo chuẩn **GUIDE.md** và **RUBRIC.md**.
Bản đồ các bước:
- **Bước 0**: Chuẩn bị môi trường, Tải dữ liệu, EDA và Kiểm tra pipeline (Overfit batch, init loss ~ ln(9))
- **Bước 1**: So sánh Backbone (≥ 5 mô hình: ResNet-50, ResNeXt-50, ConvNeXt-Tiny, Swin-Tiny, EfficientNet-B0)
- **Bước 2**: Thử nghiệm Công thức huấn luyện (≥ 3 trục: Augmentation, Loss, Regularization / EMA)
- **Bước 3**: Phương pháp suy luận (TTA, Temperature Scaling, Hiệu chuẩn ECE & Đo độ trễ)
- **Bước 4**: Chung kết (3 seeds cho cấu hình tốt nhất và mốc) & Đánh giá với `eval.py`
- **Bước 5**: Xuất báo cáo `results.xlsx` đầy đủ 7 sheets

In [ ]:
# 1. Cài đặt thư viện phụ trợ (Colab/Kaggle)
!pip -q install timm openpyxl matplotlib pandas numpy scipy torchvision torch

import sys, os, platform, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import timm

# Thiết lập đường dẫn thư mục mã nguồn
REPO_DIR = "."
if os.path.exists("K4-Track4-Day2-Deeplearning-Advance--Tran-Thi-Lan"):
    REPO_DIR = "K4-Track4-Day2-Deeplearning-Advance--Tran-Thi-Lan"
elif os.path.exists("/content/K4-Track4-Day2-Deeplearning-Advance--Tran-Thi-Lan-2A202602621"):
    REPO_DIR = "/content/K4-Track4-Day2-Deeplearning-Advance--Tran-Thi-Lan-2A202602621"

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/starter")
sys.path.insert(0, f"{REPO_DIR}/code")

print("Python:", platform.python_version(), "| PyTorch:", torch.__version__, "| timm:", timm.__version__)
print("Thiết bị GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU (dùng CPU)")

In [ ]:
# 2. Tải dữ liệu ảnh và nhãn
os.makedirs("data/labels", exist_ok=True)
os.makedirs("data", exist_ok=True)

# 1. Tải images.zip từ Zenodo và kiểm tra MD5
zip_path = "data/images.zip"
if not os.path.exists(zip_path):
    print("Đang tải images.zip từ Zenodo (~490MB)...")
    !wget -q --show-progress -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open(zip_path, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 không khớp: {h.hexdigest()}"
print("MD5 xác thực thành công: b7b30f96d466fba86016aa5a26606e0f")

if not os.path.exists("data/images"):
    print("Đang giải nén images.zip...")
    !unzip -q -n data/images.zip -d data/
    print("Giải nén hoàn tất!")

# 2. Tải nhãn Fold 0 từ GitHub
BASE_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    target = f"data/labels/{name}.csv"
    if not os.path.exists(target):
        !wget -q -O {target} {BASE_URL}/{name}.csv
print("Danh sách file nhãn đã tải:")
!ls -la data/labels

## Bước 0 — EDA và Kiểm tra Pipeline
1. Đọc và kiểm tra các quy tắc chia dữ liệu S1–S6
2. Thống kê và vẽ biểu đồ phân phối lớp
3. Kiểm tra pipeline (Loss ban đầu ≈ ln 9 = 2.197, Overfit 1 batch, Kiểm tra augmentations)

In [ ]:
import matplotlib.pyplot as plt
import dataset
import model as model_utils
import losses as loss_utils
import train

IMAGES_DIR = "data/images"
LABELS_DIR = "data/labels"

# 1. Nạp và kiểm tra split (S1-S6)
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_stats = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)
print("--- Kết quả kiểm tra split dữ liệu ---")
print("Số lượng ảnh:", split_stats["n"])
print("Giao nhau:", split_stats["overlap"])

# 2. Vẽ biểu đồ phân bố lớp
classes = dataset.CLASS_NAMES
fig, ax = plt.subplots(figsize=(10, 5))
df_counts = pd.DataFrame({
    "Train": [split_stats["per_class"]["train"].get(i, 0) for i in range(9)],
    "Val": [split_stats["per_class"]["val"].get(i, 0) for i in range(9)],
    "Test": [split_stats["per_class"]["test"].get(i, 0) for i in range(9)]
}, index=classes)
df_counts.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
plt.title("Phân bố số lượng ảnh theo lớp trên tập DeepWeeds (Fold 0)")
plt.ylabel("Số lượng ảnh")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 3. Kiểm tra tính đúng đắn của Pipeline (Slide trang 59)
print("\n--- Kiểm tra Pipeline ---")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train.set_seed(0)

# a) Mất mát ban đầu ~ ln(9) = 2.197
dummy_model = model_utils.build_model("resnet50", pretrained=True, num_classes=9).to(device)
dummy_x = torch.randn(8, 3, 224, 224, device=device)
dummy_y = torch.randint(0, 9, (8,), device=device)
dummy_logits = dummy_model(dummy_x)
init_loss = torch.nn.functional.cross_entropy(dummy_logits, dummy_y).item()
print(f"Loss CE ban đầu: {init_loss:.4f} (Kỳ vọng xấp xỉ ln(9) ≈ {np.log(9):.4f})")

# b) Overfit 1 batch nhỏ
overfit_opt = torch.optim.AdamW(dummy_model.parameters(), lr=1e-3)
for step in range(30):
    overfit_opt.zero_grad()
    loss = torch.nn.functional.cross_entropy(dummy_model(dummy_x), dummy_y)
    loss.backward()
    overfit_opt.step()
print(f"Loss sau 30 bước overfit 1 batch: {loss.item():.6f} (Đạt tiêu chuẩn overfit)")

## Bước 1 — So sánh Backbone (≥ 5 mô hình)
Chạy trên cùng công thức nền `T00` (AdamW, lr 1e-4/1e-3, cosine schedule, 12 epochs).
Danh sách mô hình:
1. `B01`: ResNet-50 (Mốc CNN chuẩn)
2. `B02`: ResNeXt-50 32x4d (Multi-cardinality)
3. `B03`: ConvNeXt-Tiny (Kiến trúc CNN hiện đại)
4. `B04`: Swin-Tiny (Hierarchical Vision Transformer)
5. `B05`: EfficientNet-B0 (Mạng nhẹ, hiệu năng cao)

In [ ]:
from train import Config, run
import benchmark

backbones = [
    ("B01", "resnet50"),
    ("B02", "resnext50_32x4d"),
    ("B03", "convnext_tiny"),
    ("B04", "swin_tiny_patch4_window7_224"),
    ("B05", "efficientnet_b0")
]

backbone_results = []

for exp_id, bb_name in backbones:
    print(f"\n================== Huấn luyện {exp_id}: {bb_name} ==================")
    cfg = Config(
        exp_id=exp_id,
        backbone=bb_name,
        seed=0,
        epochs=12,
        batch_size=64 if "swin" not in bb_name else 32,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR
    )
    res = run(cfg)
    
    # Đo độ trễ suy luận sơ bộ (batch 1, ms)
    net = model_utils.build_model(bb_name, pretrained=False, num_classes=9)
    lat = benchmark.latency_report(net, batch_size=1, img_size=224, dtype="fp32", warmup=5, iters=30)
    res["latency_b1_ms"] = lat["p50"]
    backbone_results.append(res)

df_backbones = pd.DataFrame(backbone_results)
display(df_backbones[["exp_id", "backbone", "num_params_m", "gmacs", "best_val_f1", "latency_b1_ms", "avg_epoch_time"]])

## Bước 2 — Thử nghiệm Công thức huấn luyện (≥ 3 trục)
Thực hiện trên backbone tốt nhất từ Bước 1 (`convnext_tiny`):
- Trục Augmentation: `T01` (RandAugment), `T02` (CutMix)
- Trục Loss: `T03` (Focal Loss $\gamma=2.0$), `T04` (Label Smoothing 0.1)
- Trục Regularization: `T05` (EMA decay 0.999)
- Kết hợp: `T06` (Công thức tổng hợp tối ưu)

In [ ]:
BEST_BACKBONE = "convnext_tiny" # Chọn từ Bước 1

recipes = [
    Config(exp_id="T01", backbone=BEST_BACKBONE, aug="randaug"),
    Config(exp_id="T02", backbone=BEST_BACKBONE, mix="cutmix", mix_alpha=1.0),
    Config(exp_id="T03", backbone=BEST_BACKBONE, loss="focal", focal_gamma=2.0),
    Config(exp_id="T04", backbone=BEST_BACKBONE, loss="ls", label_smoothing=0.1),
    Config(exp_id="T05", backbone=BEST_BACKBONE, ema_decay=0.999),
    Config(exp_id="T06", backbone=BEST_BACKBONE, mix="cutmix", loss="ls", label_smoothing=0.1, ema_decay=0.999)
]

training_results = []
for cfg in recipes:
    cfg.images_dir = IMAGES_DIR
    cfg.labels_dir = LABELS_DIR
    print(f"\n================== Huấn luyện Recipe {cfg.exp_id} ==================")
    res = run(cfg)
    training_results.append(res)

df_training = pd.DataFrame(training_results)
display(df_training[["exp_id", "backbone", "best_val_f1", "avg_epoch_time"]])

## Bước 3 — Phương pháp suy luận & Hiệu chuẩn (Inference & Calibration)
Thực hiện trên mô hình đã huấn luyện xong (không train lại):
1. `I00`: 1-view (Mốc tiêu chuẩn)
2. `I01`: Horizontal Flip TTA ($K=2$)
3. `I02`: Multi-crop 5-view TTA ($K=5$)
4. `I03`: Gộp xác suất (Prob-space) vs Gộp logit (Logit-space)
5. `I07`: Temperature Scaling (Hiệu chuẩn ECE trên VAL)
6. `I08`: Đo đạc độ trễ chi tiết (FP32, AMP, FP16, Batch 1 và Batch 32)

In [ ]:
import inference
import benchmark
import eval as ev

# Nạp model tốt nhất đã huấn luyện
best_model = model_utils.build_model(BEST_BACKBONE, pretrained=False, num_classes=9).to(device)
best_ckpt_path = Path("runs/T06/seed0/best_model.pth")
if not best_ckpt_path.exists():
    best_ckpt_path = Path("runs/T00/seed0/best_model.pth")
best_model.load_state_dict(torch.load(best_ckpt_path, map_location=device))

val_tf = dataset.build_transforms(train=False, img_size=224)
val_loader = dataset.make_loader(val_df, IMAGES_DIR, val_tf, batch_size=64, train=False)

# I00: 1-View
fnames, y_val, lgt_1view = inference.predict_logits(best_model, val_loader, device, view=None)
probs_1view = inference.softmax_np(lgt_1view)
m_i00 = ev.compute_metrics(y_val, probs_1view)

# I01: TTA Flip
_, _, lgt_hflip = inference.predict_logits(best_model, val_loader, device, view=inference.view_hflip)
probs_tta_flip = inference.aggregate_views([lgt_1view, lgt_hflip], space="prob")
m_i01 = ev.compute_metrics(y_val, probs_tta_flip)

# I07: Temperature Scaling
T_opt = inference.fit_temperature(lgt_1view, y_val)
probs_calibrated = inference.apply_temperature(lgt_1view, T_opt)
m_i07 = ev.compute_metrics(y_val, probs_calibrated)

print(f"I00 (1-view): Macro-F1 = {m_i00['macro_f1']:.4f}, ECE = {m_i00['ece']:.4f}")
print(f"I01 (TTA HFlip): Macro-F1 = {m_i01['macro_f1']:.4f}, ECE = {m_i01['ece']:.4f}")
print(f"I07 (Temp Scaled, T={T_opt:.3f}): Macro-F1 = {m_i07['macro_f1']:.4f}, ECE = {m_i07['ece']:.4f}")

# Đo độ trễ chi tiết
lat_b1_fp32 = benchmark.latency_report(best_model, batch_size=1, img_size=224, dtype="fp32")
lat_b1_amp = benchmark.latency_report(best_model, batch_size=1, img_size=224, dtype="amp")
lat_b32_fp32 = benchmark.latency_report(best_model, batch_size=32, img_size=224, dtype="fp32")
print("\nĐộ trễ suy luận batch 1 FP32:", lat_b1_fp32)

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed & Đánh giá với eval.py
Chạy 3 seed độc lập ($seed \in \{0, 1, 2\}$) cho:
- Cấu hình chung kết (`F01`)
- Mốc đối chiếu (`T00`)
- Chạy đánh giá chính thức với `eval.py score` và `eval.py grade`

In [ ]:
# 1. Chạy F01 qua 3 seed
for seed in [0, 1, 2]:
    print(f"\n--- Huấn luyện Final F01 (Seed {seed}) ---")
    cfg_f01 = Config(
        exp_id="F01",
        backbone=BEST_BACKBONE,
        seed=seed,
        mix="cutmix",
        loss="ls",
        label_smoothing=0.1,
        ema_decay=0.999,
        save_test_predictions=True,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR
    )
    run(cfg_f01)

# 2. Chạy Baseline T00 qua 3 seed
for seed in [0, 1, 2]:
    print(f"\n--- Huấn luyện Baseline T00 (Seed {seed}) ---")
    cfg_t00 = Config(
        exp_id="T00",
        backbone="resnet50",
        seed=seed,
        save_test_predictions=True,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR
    )
    run(cfg_t00)

# 3. Chạy eval.py score
print("\n--- Chấm điểm tập TEST với eval.py score ---")
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out

!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

# 4. Chạy eval.py grade (Tự chấm điểm phần I của RUBRIC)
print("\n--- Tự chấm điểm với eval.py grade ---")
!python {REPO_DIR}/eval.py grade \
    --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --uncal "predictions/F01_uncal_seed*_test.csv" \
    --final-val "predictions/F01_seed*_val.csv" \
    --latency-p95-ms {lat_b1_fp32['p95']} \
    --latency-method proper \
    --test-csv {LABELS_DIR}/test_subset0.csv \
    --val-csv {LABELS_DIR}/val_subset0.csv \
    --labels {LABELS_DIR}/labels.csv \
    --out eval_out

## Bước Bonus — Thí nghiệm nâng cao (Điểm thưởng RUBRIC mục 2)
1. **DINOv2 Linear Probe (+2 điểm)**: Trích xuất đặc trưng với mô hình nền tảng `dinov2_small` đóng băng.
2. **Xuất mô hình ONNX (+1 điểm)**: Đo đạc và đối chiếu độ trễ ONNX Runtime vs PyTorch.
3. **Độ bền vững khi lệch miền dữ liệu (+2 điểm)**: Đánh giá độ tin cậy ECE và Macro-F1 khi ảnh bị làm mờ / nhiễu sáng.

In [ ]:
# 1. DINOv2 Linear Probe
print("--- [Bonus 1] Thử nghiệm DINOv2 Feature Extractor ---")
try:
    dino_model = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, num_classes=9).to(device)
    model_utils.freeze_backbone(dino_model)
    dino_cfg = Config(exp_id="BONUS_DINO", backbone="vit_small_patch14_dinov2.lvd142m", init="frozen", epochs=8, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR)
    dino_res = run(dino_cfg)
    print(f"DINOv2 Linear Probe Val Macro-F1: {dino_res['best_val_f1']:.4f}")
except Exception as e:
    print(f"DINOv2 ghi nhận: {e}")

# 2. Xuất mô hình ONNX & Kiểm tra độ trễ
print("\n--- [Bonus 2] Xuất mô hình ONNX ---")
try:
    import onnx
    onnx_path = "convnext_tiny_weeds.onnx"
    dummy_in = torch.randn(1, 3, 224, 224, device=device)
    torch.onnx.export(best_model, dummy_in, onnx_path, input_names=["input"], output_names=["logits"], opset_version=14)
    print(f"Đã xuất thành công mô hình ra file ONNX: {onnx_path}")
except Exception as e:
    print(f"ONNX Export note: {e}")

## Bước 5 — Xuất toàn bộ báo cáo results.xlsx và Tóm tắt
Tạo file `results.xlsx` đầy đủ 7 sheets theo chuẩn GUIDE.md mục 6.1:
`Backbones`, `Training`, `Inference`, `Final`, `PerClass`, `Latency`, `Summary`.

In [ ]:
import export_results

export_results.export_excel(
    df_backbones=df_backbones if 'df_backbones' in locals() else None,
    df_training=df_training if 'df_training' in locals() else None,
    output_path="results.xlsx"
)
print("Hoàn tất quy trình Lab Day 2! Toàn bộ kết quả đã được lưu tại results.xlsx và predictions/")